# FormCoach — Model A training on Colab GPU
Runtime → Change runtime type → GPU. Data lives in Google Drive under `formcoach/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA='/content/drive/MyDrive/formcoach'

In [ ]:
!git clone https://github.com/YOUR_ORG/formcoach-models.git
%cd formcoach-models
!pip -q install mediapipe google-genai
!python -m pytest -q tests

## 1. Extract reps + crops (skip if `processed/` already exists)
Then open a few QA plots in `processed/qa/`.

In [ ]:
!python scripts/01_extract_reps.py --videos $DATA/raw --out $DATA/processed
!python scripts/02_make_splits.py --data $DATA/processed --pin p09=test

## 2. Train: pretrained baseline (frozen backbone) and fine-tuned model

In [ ]:
!python scripts/03_train_model_a.py --data $DATA/processed --mode probe --out $DATA/runs/probe
!python scripts/03_train_model_a.py --data $DATA/processed --mode finetune --out $DATA/runs/finetune

## 3. Evaluate on unseen participants + skeleton-only baseline

In [ ]:
!python scripts/04_eval_model_a.py --data $DATA/processed --runs $DATA/runs/probe $DATA/runs/finetune --export-preds $DATA/runs/finetune
!python scripts/04b_feature_baseline.py --data $DATA/processed
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{DATA}/runs/results_model_a.md').read()))
for f in ['curves.png','confusion.png','success.jpg','failure.jpg']:
    display(Image(f'{DATA}/runs/finetune/{f}'))

## 4. Export TFLite for the app

In [ ]:
!python scripts/05_export_tflite.py --run $DATA/runs/finetune --data $DATA/processed --fp16